# 02 · Phishing Guard and Coach

A two-agent assistant on an employee's Outlook inbox at Harbor Foods, a fictional food distributor. **Guard** investigates each new email with tools, records a verdict, and asks the employee before moving anything. **Coach** teaches the employee when they call it wrong.

All logic lives in the `phishing_guard` package. Each section below imports the piece it needs and shows it working. Sections follow PLAN.md: Setup, World, Classifier, Tools, Agents, Runners, Demo, Evaluation, Results, Findings.

**Status: Monday, Oct 5.** Sections 1 to 7 are built. Guard has all nine tools, the approval pause works both ways, and both demo cases produce the expected trace twice in a row (`scripts/monday_check.py`). Evaluation, Results, and Findings arrive Tuesday.

## 1. Setup

The OpenAI key is read from the environment via python-dotenv (`.env` is never printed). If it is missing, `ensure_openai_key()` asks with a hidden prompt. The model is `gpt-5.4-mini`, as in the class tutorial, and tracing is off.

In [ ]:
import json
from phishing_guard.config import ensure_openai_key, has_openai_key, lm_model

ensure_openai_key()
print("model:", lm_model, "| key loaded:", has_openai_key())

## 2. World

Five Python structures hold all state: `USERS`, `MAILBOX`, `THREAT_INTEL`, `TRAINING_LOG`, `SECURITY_QUEUE`. `reset_world()` restores them before every run so each case starts clean. `TEST_CASES` is the answer key; no tool reads it. Mailbox and threat intel load from `data/*.json`.

In [ ]:
from phishing_guard.world import (USERS, MAILBOX, THREAT_INTEL, TRAINING_LOG,
                                  SECURITY_QUEUE, TEST_CASES, reset_world)

reset_world()
print(f"{len(USERS)} users, {len(MAILBOX)} emails, "
      f"{len(THREAT_INTEL['domains'])} domain rows, {len(THREAT_INTEL['urls'])} url rows, "
      f"{len(TEST_CASES)} test cases")
for uid, u in USERS.items():
    print(f"  {uid} {u['name']:<12} {u['role']:<18} knows {u['known_domains']}")

## 3. Classifier

Loads `phish_clf.joblib` trained in notebook 01. `top_words` lists the words that pushed an email's score toward phishing, which Guard sees as part of the tool result. Below, the classifier alone on the ten development emails: it should flag the obvious ones and miss the social-engineering ones that have no classic phishing vocabulary.

In [ ]:
from phishing_guard.classifier import load_classifier, top_words

clf = load_classifier()
truth = {c['email_id']: c['true_label'] for c in TEST_CASES.values()}
print(f"{'email':<7}{'p(phish)':<10}{'truth':<12}top words")
for eid, e in sorted(MAILBOX.items()):
    text = e['subject'] + ' ' + e['body']
    p = float(clf.predict_proba([text])[0][1])
    print(f"{eid:<7}{p:<10.2f}{truth[eid]:<12}{top_words(text)}")

## 4. Tools

Guard's tools are `@function_tool` functions in `phishing_guard/tools.py`. Four gather evidence, one records the verdict, one acts on the mailbox behind a human approval, and two follow up. The ninth tool, `coach_user`, is the Coach agent wrapped with `.as_tool()` in section 5.

`check_sender_domain` and `check_url` take the value itself, so Guard has to extract the right domain or link, which is what the argument-correctness metric checks. Every tool returns data on failure and never raises.

`move_to_phishing_folder` has `needs_approval=True`, so the SDK pauses before running it, and a `@tool_input_guardrail` named `check_move` (the tutorial's `check_refund` pattern) blocks the move when the email ID is unknown, no verdict is recorded, or the verdict is legitimate. The guardrail runs after approval, right before execution, so approval cannot override it.

In [ ]:
from phishing_guard.tools import FUNCTION_TOOLS

for t in FUNCTION_TOOLS:
    params = list(t.params_json_schema['properties'])
    flags = []
    if t.needs_approval:
        flags.append('needs approval')
    if t.tool_input_guardrails:
        flags.append('guardrail: check_move')
    print(f"{t.name:<24} {params!s:<40} {', '.join(flags)}")

## 5. Agents

Coach is defined first because Guard's tool list references it, the same pattern as the tutorial's NASA manager and translator. Coach has one job: turn Guard's evidence into a lesson of 120 words or fewer for one employee about one email.

Guard's instructions follow five labeled stages (INVESTIGATE, DECIDE, ACT, FOLLOW UP, REPLY) so the trace can be checked against them. `make_guard_full()` builds Guard with all nine tools in PLAN.md order. `build_input` turns a mailbox record into the message Guard receives; it leaves out `is_simulation` and every label.

Two wording fixes were made on Monday after the first end-to-end runs, one change each, and are noted in the source: FOLLOW UP now says the move result carries the employee's decision and Guard must not reply before the stage is done; REPLY now says to copy the Coach's lesson exactly.

In [ ]:
from phishing_guard.guard_agents import make_coach, make_guard_full, build_input

coach = make_coach()
guard = make_guard_full(coach)
print("Guard tools:", [t.name for t in guard.tools])
print()
print(build_input("E-002"))

## 6. Runners

`show_trace` is the tutorial's helper for reading `result.new_items`. Three runners in `phishing_guard/runners.py`:

- `run_investigation(guard, email_id)`: one run, no pause handling. Sunday's check.
- `run_interactive(guard, email_id)`: the live demo. When Guard asks to move an email, a person types approve or reject, and on reject, their reason.
- `run_case(guard, case)`: evaluation. The decision and reason come from the test case, so 30 cases run without typing.

Both approval runners stop after three pauses and return every result segment. The SDK repeats pre-pause items after a resume (`scripts/probe_sdk.py`), so `collect_calls` in `phishing_guard/evaluation.py` dedupes by `call_id`.

**Sunday's end-of-day check, kept for the record.** Guard with only the five investigate/decide tools records a sensible verdict on demo cases A, B, and C.

In [ ]:
from agents import Runner
from phishing_guard.trace_utils import show_trace
from phishing_guard.runners import run_investigation, run_interactive, run_case, recorded_verdict
from phishing_guard.guard_agents import make_guard, SUNDAY_ADDENDUM
from phishing_guard.tools import SUNDAY_TOOLS

guard_sunday = make_guard(SUNDAY_TOOLS, SUNDAY_ADDENDUM)
expected = {c['email_id']: c['expected_verdict'] for c in TEST_CASES.values()}
summary = []
for eid in ["E-001", "E-002", "E-003"]:
    print("=" * 72, f"\n{eid}  expected: {expected[eid]}\n" + "=" * 72)
    result = await run_investigation(guard_sunday, eid)
    show_trace(result)
    rv = recorded_verdict(eid)
    print("\nRecorded:", rv)
    summary.append((eid, expected[eid], rv['verdict']))

print("\nSUMMARY")
for eid, exp, rec in summary:
    print(f"{eid}  expected={exp:<12} recorded={rec}")

## 7. Demo

Two live cases show both branches of the approval. Each cell asks for a decision at the pause.

| Case | Email | Employee does | Proves |
| --- | --- | --- | --- |
| A | E-001, fake Microsoft 365 password-expiry notice to Priya with a blocklisted link | **approve** | Tool choice, sequence, approval gate. Email moves to Phishing, decision logged as reported. |
| B | E-002, bank-detail change to Dana from `apex-supp1y.example`, one character off her real vendor, no links, simulation flag on | **reject**, reason: That's Apex, we pay them every month. | No link check. Profile shows the real vendor domain. Coach names the swapped character and the reply-to mismatch. Logged as missed, security ticket, simulation reveal. |
| C (optional) | E-003, real invoice from `apexsupply.example` | not asked | Guard clears it with no action. |

`scripts/monday_check.py --repeat 2` runs A and B through `run_case` with scripted decisions and checks the trace against the test case and the HITL table. On Monday it passed 4/4.

In [ ]:
# Demo case A: type approve at the pause.
results_a = await run_interactive(guard, "E-001")
for r in results_a:
    show_trace(r)
print("\nFolder:", MAILBOX["E-001"]["folder"], "| training log:", TRAINING_LOG, "| security queue:", SECURITY_QUEUE)

In [ ]:
# Demo case B: type reject at the pause, then the reason: That's Apex, we pay them every month.
results_b = await run_interactive(guard, "E-002")
for r in results_b:
    show_trace(r)
print("\nFolder:", MAILBOX["E-002"]["folder"], "| training log:", TRAINING_LOG, "| security queue:", SECURITY_QUEUE)

In [ ]:
# Demo case C (optional): Guard should clear it with no pause.
results_c = await run_interactive(guard, "E-003")
print("Folder:", MAILBOX["E-003"]["folder"], "| pauses:", sum(bool(r.interruptions) for r in results_c))

In [ ]:
# The same two cases with scripted decisions, as the evaluation will run them.
from phishing_guard.evaluation import collect_calls

for case_id in ["C-01", "C-02"]:
    case = TEST_CASES[case_id]
    results = await run_case(guard, case)
    calls = collect_calls(results)
    print(f"{case_id}: {[c['name'] for c in calls]}")
    print("   folder:", MAILBOX[case['email_id']]['folder'],
          "| logged:", [r['outcome'] for r in TRAINING_LOG],
          "| escalated:", [q['email_id'] for q in SECURITY_QUEUE])

## 8. Evaluation · 9. Results · 10. Findings

Built Tuesday per PLAN.md: `score_case`, the 30-case run loop, the classifier-only baseline, the results table, and the lesson judge.